In [1]:
%pip install wget pandas dash

  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached werkzeug-3.1.9-py3-none-any.whl.metadata (4.1 kB)
  Using cached plotly-7.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached retrying-1.4.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached janus-2.0.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 26.5 MB/s  0:00:00 eta 0:00:01
Using cached flask-3.1.3-py3-none-any.whl (103 kB)
Using cached werkzeug-3.1.9-py3-none-any.whl (228 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached itsdangerous-2.2.0-py3-none-any.whl (16 kB)
Using cached janus-2.0.0-py3-none-any.whl (12 kB)
Using cached plotly-7.1.0-py3-none-any.whl (9.7 MB)
   ━━━━

In [1]:
import wget
import pandas as pd

url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'

spacex_csv_file = wget.download(url, out='/tmp/spacex_launch_geo.csv')

spacex_df = pd.read_csv(spacex_csv_file)

spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [2]:
print(spacex_df['Launch Site'].unique())
print(spacex_df['Launch Site'].value_counts())

<StringArray>
['CCAFS LC-40', 'VAFB SLC-4E', 'KSC LC-39A', 'CCAFS SLC-40']
Length: 4, dtype: str
Launch Site
CCAFS LC-40     26
KSC LC-39A      13
VAFB SLC-4E     10
CCAFS SLC-40     7
Name: count, dtype: int64


In [3]:
spacex_df.loc[
    spacex_df['Launch Site'].str.contains('CCAFS', na=False),
    'Launch Site'
].value_counts()

Launch Site
CCAFS LC-40     26
CCAFS SLC-40     7
Name: count, dtype: int64

In [4]:
import dash
from dash import html
from dash import dcc
from dash.dependencies import Input, Output
import plotly.express as px

In [5]:

max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

In [6]:
# Create a dash application
app = dash.Dash(__name__)

In [7]:
app.layout = html.Div(children=[
    html.H1(
        'SpaceX Launch Records Dashboard',
        style={
            'textAlign': 'center',
            'color': '#503D36',
            'font-size': 40
        }
    ),

dcc.Dropdown(
    id='site-dropdown',
    options=[
        {'label': 'All Sites', 'value': 'ALL'},
        {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'},
        {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'},
        {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'},
        {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}
    ],
    value='ALL',
    placeholder='Select a Launch Site here',
    searchable=True),
                    html.Br(),
                      html.Div(
                           dcc.Graph(id='success-pie-chart')),
                           html.Br(),
                           html.P('Payload range (Kg):'),
                           dcc.RangeSlider(
                               id='payload-slider',
                               min=0,
                               max=10000,
                               step=1000,
                               marks={ 0: '0', 10000: '10000'},
                               value=[min_payload, max_payload]),

    html.Div(
        dcc.Graph(id='success-payload-scatter-chart')
    )
])

In [8]:
# TASK 2:
# Add a callback function for `site-dropdown` as input, `success-pie-chart` as output
@app.callback(
    Output(
        component_id='success-pie-chart',
        component_property='figure' ),
    Input(
        component_id='site-dropdown',
        component_property='value') )
def get_pie_chart(entered_site):
    filtered_df = spacex_df
    if entered_site == 'ALL':
        fig = px.pie(
        filtered_df,
        values='class',
        names='Launch Site',
        title='Total Success Launches By Site ')
        return fig
    else:
        filtered_df = spacex_df[ spacex_df['Launch Site'] == entered_site ]
        fig = px.pie(
        filtered_df,
        
        names='class',
        title=f'Total Success Launches for Site {entered_site}' )
        return fig

In [9]:
# TASK 4:
# Add a callback function for `site-dropdown` and `payload-slider` as inputs, `success-payload-scatter-chart` as output

@app.callback(
    Output(
        component_id='success-payload-scatter-chart',
        component_property='figure'
    ),
    Input(
        component_id='site-dropdown',
        component_property='value'
    ),
    Input(
        component_id='payload-slider',
        component_property='value'
    )
)
def get_scatter_chart(entered_site, payload_range):

    low, high = payload_range

    filtered_df = spacex_df[
        (spacex_df['Payload Mass (kg)'] >= low) &
        (spacex_df['Payload Mass (kg)'] <= high)
    ]

    if entered_site == 'ALL':

        fig = px.scatter(
            filtered_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version',
            title='Correlation between Payload and Success for all Site'
        )

    else:

        filtered_df = filtered_df[
            filtered_df['Launch Site'] == entered_site
        ]

        fig = px.scatter(
            filtered_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version',
            title=f'Correlation between Payload and Success for  Site {entered_site}'
        )

    return fig


In [10]:
# Run the app
if __name__ == '__main__':
    app.run()